# Notebook 3 – Porto Seguro Safe Driver Prediction (Classification)

## Overview
We benchmark five tabular-learning methods on the Porto Seguro dataset:
**Tabular ResNet**, **FT-Transformer**, **XGBoost**, **LightGBM**, and **Random Forest**.  
Each model is tuned with **Optuna** (20 trials) and evaluated across 3 seeds.  
Metrics: **Accuracy**, **AUC-ROC**, **Normalized Gini** (= 2·AUC − 1), **F1**.

> **Note:** The dataset is heavily imbalanced (~3.6 % positives).  
> If the Kaggle API is not configured, a synthetic dataset matching the Porto
> Seguro schema is generated automatically.


In [1]:
# !pip install "rtdl==0.0.13" optuna xgboost lightgbm ucimlrepo scikit-learn pandas numpy matplotlib seaborn shap

## Imports

In [15]:
import warnings
warnings.filterwarnings('ignore')

import random, os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

import rtdl_revisiting_models
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [3]:
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.backends.cudnn.version())

True
12.1
90100


## Configuration

In [4]:
SEEDS           = [42, 123, 456]
N_OPTUNA_TRIALS = 20
TEST_SIZE       = 0.20
VAL_FRAC        = 0.25


## Data Loading

In [5]:
os.makedirs('data/porto_seguro', exist_ok=True)
df = None
zip_path = 'data/porto-seguro-safe-driver-prediction.zip'

try:
    # Open the existing ZIP file and extract
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('data/porto_seguro')

    # Load the CSV directly
    csv_path = 'data/porto_seguro/train.csv'
    df = pd.read_csv(csv_path)
    print(f"Loaded real data: {df.shape}")

except Exception as e:
    print(f"Failed to read from local ZIP: {e}")
    print("Please make sure the file exists at:")
    print(zip_path)
    df = None

Loaded real data: (595212, 59)


## EDA

In [6]:
print("Shape:", df.shape)
print()
print("Target distribution:")
print(df['target'].value_counts())
print(f"Positive rate: {df['target'].mean():.3%}")


Shape: (595212, 59)

Target distribution:
target
0    573518
1     21694
Name: count, dtype: int64
Positive rate: 3.645%


In [7]:
df.columns

Index(['id', 'target', 'ps_ind_01', 'ps_ind_02_cat', 'ps_ind_03',
       'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_ind_06_bin', 'ps_ind_07_bin',
       'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_10_bin', 'ps_ind_11_bin',
       'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14', 'ps_ind_15',
       'ps_ind_16_bin', 'ps_ind_17_bin', 'ps_ind_18_bin', 'ps_reg_01',
       'ps_reg_02', 'ps_reg_03', 'ps_car_01_cat', 'ps_car_02_cat',
       'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat', 'ps_car_06_cat',
       'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat',
       'ps_car_11_cat', 'ps_car_11', 'ps_car_12', 'ps_car_13', 'ps_car_14',
       'ps_car_15', 'ps_calc_01', 'ps_calc_02', 'ps_calc_03', 'ps_calc_04',
       'ps_calc_05', 'ps_calc_06', 'ps_calc_07', 'ps_calc_08', 'ps_calc_09',
       'ps_calc_10', 'ps_calc_11', 'ps_calc_12', 'ps_calc_13', 'ps_calc_14',
       'ps_calc_15_bin', 'ps_calc_16_bin', 'ps_calc_17_bin', 'ps_calc_18_bin',
       'ps_calc_19_bin', 'ps_calc_20_bin'],


In [8]:
# Print the number of unique values in each column
print("Number of unique values in each column:")
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")


Number of unique values in each column:
id: 595212
target: 2
ps_ind_01: 8
ps_ind_02_cat: 5
ps_ind_03: 12
ps_ind_04_cat: 3
ps_ind_05_cat: 8
ps_ind_06_bin: 2
ps_ind_07_bin: 2
ps_ind_08_bin: 2
ps_ind_09_bin: 2
ps_ind_10_bin: 2
ps_ind_11_bin: 2
ps_ind_12_bin: 2
ps_ind_13_bin: 2
ps_ind_14: 5
ps_ind_15: 14
ps_ind_16_bin: 2
ps_ind_17_bin: 2
ps_ind_18_bin: 2
ps_reg_01: 10
ps_reg_02: 19
ps_reg_03: 5013
ps_car_01_cat: 13
ps_car_02_cat: 3
ps_car_03_cat: 3
ps_car_04_cat: 10
ps_car_05_cat: 3
ps_car_06_cat: 18
ps_car_07_cat: 3
ps_car_08_cat: 2
ps_car_09_cat: 6
ps_car_10_cat: 3
ps_car_11_cat: 104
ps_car_11: 5
ps_car_12: 184
ps_car_13: 70482
ps_car_14: 850
ps_car_15: 15
ps_calc_01: 10
ps_calc_02: 10
ps_calc_03: 10
ps_calc_04: 6
ps_calc_05: 7
ps_calc_06: 11
ps_calc_07: 10
ps_calc_08: 11
ps_calc_09: 8
ps_calc_10: 26
ps_calc_11: 20
ps_calc_12: 11
ps_calc_13: 14
ps_calc_14: 24
ps_calc_15_bin: 2
ps_calc_16_bin: 2
ps_calc_17_bin: 2
ps_calc_18_bin: 2
ps_calc_19_bin: 2
ps_calc_20_bin: 2


In [9]:
# -1 values as missing indicator
n_neg1 = (df == -1).sum()
print("Columns with -1 (missing values):")
print(n_neg1[n_neg1 > 0])


Columns with -1 (missing values):
ps_ind_02_cat       216
ps_ind_04_cat        83
ps_ind_05_cat      5809
ps_reg_03        107772
ps_car_01_cat       107
ps_car_02_cat         5
ps_car_03_cat    411231
ps_car_05_cat    266551
ps_car_07_cat     11489
ps_car_09_cat       569
ps_car_11             5
ps_car_12             1
ps_car_14         42620
dtype: int64


In [13]:
# Find columns with high collinearity
corr_matrix = df.drop(columns=['id', 'target']).corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
collinear_pairs = [
    (col, row, upper_tri.loc[row, col])
    for col in upper_tri.columns
    for row in upper_tri.index
    if upper_tri.loc[row, col] > 0.9
]
print("Highly correlated column pairs (corr > 0.9):")
for pair in collinear_pairs:
    print(f"{pair[0]} - {pair[1]} : {pair[2]:.2f}")


Highly correlated column pairs (corr > 0.9):


## Preprocessing

In [14]:
df_proc = df.drop(columns=['id']).copy()
y = df_proc['target'].values.astype(np.int64)
df_feat = df_proc.drop(columns=['target'])

cat_cols_ps  = [c for c in df_feat.columns if c.endswith('_cat')]
num_cols_ps  = [c for c in df_feat.columns if c not in cat_cols_ps]

print(f"Numerical features: {len(num_cols_ps)}")
print(f"Categorical features: {len(cat_cols_ps)}")

# Replace -1 with NaN
df_feat[cat_cols_ps] = df_feat[cat_cols_ps].replace(-1, np.nan)
df_feat[num_cols_ps] = df_feat[num_cols_ps].replace(-1, np.nan)

# Impute
for col in cat_cols_ps:
    mode_val = df_feat[col].mode()
    if len(mode_val) > 0:
        df_feat[col] = df_feat[col].fillna(mode_val[0])
    else:
        df_feat[col] = df_feat[col].fillna(0)

for col in num_cols_ps:
    df_feat[col] = df_feat[col].fillna(df_feat[col].median())

print("Missing after imputation:", df_feat.isnull().sum().sum())


Numerical features: 43
Categorical features: 14
Missing after imputation: 0


In [17]:
# Define a threshold for "too many categories"
max_cardinality = 50   # adjust as needed

# Compute cardinalities
cat_cardinalities = df_feat[cat_cols_ps].nunique()

# Keep only categorical columns below threshold
cat_cols_filtered = [col for col in cat_cols_ps if cat_cardinalities[col] <= max_cardinality]

print("Dropped high-cardinality columns:", [col for col in cat_cols_ps if col not in cat_cols_filtered])
print("Kept categorical columns:", cat_cols_filtered)

# Ordinal-encode the filtered categorical columns (integer indices for neural embeddings)
ordinal_enc_ps = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_cat_ps = ordinal_enc_ps.fit_transform(df_feat[cat_cols_filtered].astype(str)).astype(np.int64)

# Cardinalities for embedding layers (number of unique categories per column)
cat_cards_ps = [int(cat_cardinalities[col]) for col in cat_cols_filtered]

# Scale numerical variables
scaler_ps = StandardScaler()
X_num_ps = scaler_ps.fit_transform(df_feat[num_cols_ps].values.astype(np.float32))

# Combine numerical (scaled) and categorical (ordinal-coded as float) arrays
# Tree-based models receive this flat float array directly
X_all_ps = np.concatenate([X_num_ps, X_cat_ps.astype(np.float32)], axis=1)
print("X_all shape:", X_all_ps.shape)
print("Categorical cardinalities:", cat_cards_ps)

Dropped high-cardinality columns: ['ps_car_11_cat']
Kept categorical columns: ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat', 'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat', 'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat']
X_all shape: (595212, 114)


## Data Splitting (60 / 20 / 20)

In [18]:
idx = np.arange(len(y))
idx_tv, idx_test = train_test_split(idx, test_size=TEST_SIZE, random_state=42, stratify=y)
idx_train, idx_val = train_test_split(idx_tv, test_size=VAL_FRAC, random_state=42,
                                       stratify=y[idx_tv])

X_tr_all, X_v_all, X_te_all = X_all_ps[idx_train], X_all_ps[idx_val], X_all_ps[idx_test]
X_tr_num, X_v_num, X_te_num = X_num_ps[idx_train], X_num_ps[idx_val], X_num_ps[idx_test]
X_tr_cat, X_v_cat, X_te_cat = X_cat_ps[idx_train], X_cat_ps[idx_val], X_cat_ps[idx_test]
y_train_ps, y_val_ps, y_test_ps = y[idx_train], y[idx_val], y[idx_test]

sc_ps = StandardScaler()
X_tr_sc = sc_ps.fit_transform(X_tr_all)
X_v_sc  = sc_ps.transform(X_v_all)
X_te_sc = sc_ps.transform(X_te_all)

spw = float((y_train_ps == 0).sum()) / float((y_train_ps == 1).sum())
print(f"Train: {X_tr_sc.shape}, Val: {X_v_sc.shape}, Test: {X_te_sc.shape}")
print(f"scale_pos_weight = {spw:.2f}")


Train: (357126, 114), Val: (119043, 114), Test: (119043, 114)
scale_pos_weight = 26.44


## Helper Functions

In [35]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_porto_metrics(y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)
    gini = 2 * auc - 1
    f1   = f1_score(y_true, y_pred, zero_division=0)
    return acc, auc, gini, f1


def train_ft_transformer(model, X_num_tr, X_cat_tr, y_tr,
                          X_num_v, X_cat_v, y_v,
                          lr=1e-3, n_epochs=100, batch_size=256,
                          task='regression', device_='cpu'):
    model = model.to(device_)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss() if task == 'regression' else nn.BCEWithLogitsLoss()

    X_num_tr_t = torch.FloatTensor(X_num_tr).to(device_)
    X_cat_tr_t = torch.LongTensor(X_cat_tr).to(device_) if X_cat_tr is not None else None
    y_tr_t     = torch.FloatTensor(y_tr.astype(np.float32)).to(device_)
    X_num_v_t  = torch.FloatTensor(X_num_v).to(device_)
    X_cat_v_t  = torch.LongTensor(X_cat_v).to(device_) if X_cat_v is not None else None
    y_v_t      = torch.FloatTensor(y_v.astype(np.float32)).to(device_)

    train_losses, val_losses = [], []
    best_val   = float('inf')
    best_state = None
    patience   = 20
    pat_cnt    = 0

    for epoch in range(n_epochs):
        model.train()
        n   = len(X_num_tr_t)
        idx_e = torch.randperm(n)
        ep_loss = 0.0
        for i in range(0, n, batch_size):
            b  = idx_e[i:i+batch_size]
            xn = X_num_tr_t[b]
            xc = X_cat_tr_t[b] if X_cat_tr_t is not None else None
            yb = y_tr_t[b]
            optimizer.zero_grad()
            out  = model(xn, xc).squeeze(-1)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            vout  = model(X_num_v_t, X_cat_v_t).squeeze(-1)
            vloss = criterion(vout, y_v_t).item()
        train_losses.append(ep_loss / n)
        val_losses.append(vloss)
        if vloss < best_val:
            best_val   = vloss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_cnt    = 0
        else:
            pat_cnt += 1
        if pat_cnt >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, train_losses, val_losses


def predict_ft_transformer(model, X_num, X_cat, device_, batch_size=512):
    model.eval()
    model   = model.to(device_)
    X_num_t = torch.FloatTensor(X_num).to(device_)
    X_cat_t = torch.LongTensor(X_cat).to(device_) if X_cat is not None else None
    preds   = []
    with torch.no_grad():
        for i in range(0, len(X_num_t), batch_size):
            xn  = X_num_t[i:i+batch_size]
            xc  = X_cat_t[i:i+batch_size] if X_cat_t is not None else None
            out = model(xn, xc).squeeze(-1)
            preds.append(out.cpu().numpy())
    return np.concatenate(preds)


## Model 1: Tabular ResNet

In [36]:
all_results = []

n_num_ps = X_tr_num.shape[1]


class ResNetWithCatEmbeddings(nn.Module):
    """Tabular ResNet with per-feature categorical embeddings."""

    def __init__(self, n_num, cat_cardinalities, emb_dim,
                 n_blocks, d_main, d_hidden, dropout_first, dropout_second):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(card + 1, emb_dim) for card in cat_cardinalities
        ])
        d_in = n_num + len(cat_cardinalities) * emb_dim
        self.resnet = rtdl_revisiting_models.ResNet.make_baseline(
            d_in=d_in, d_out=1,
            n_blocks=n_blocks, d_main=d_main, d_hidden=d_hidden,
            dropout_first=dropout_first, dropout_second=dropout_second,
        )

    def forward(self, x_num, x_cat):
        embs = [self.embeddings[j](x_cat[:, j]) for j in range(x_cat.shape[1])]
        x = torch.cat([x_num] + embs, dim=1)
        return self.resnet(x)


def resnet_porto_objective(trial):
    n_blocks  = trial.suggest_int('n_blocks', 1, 4)
    d_main    = trial.suggest_categorical('d_main', [64, 128, 256])
    d_hidden  = trial.suggest_categorical('d_hidden', [128, 256, 512])
    dropout   = trial.suggest_float('dropout', 0.0, 0.5)
    emb_dim   = trial.suggest_categorical('emb_dim', [4, 8, 16])
    lr        = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    set_seed(42)
    model = ResNetWithCatEmbeddings(
        n_num=n_num_ps, cat_cardinalities=cat_cards_ps, emb_dim=emb_dim,
        n_blocks=n_blocks, d_main=d_main, d_hidden=d_hidden,
        dropout_first=dropout, dropout_second=dropout,
    )
    model, _, _ = train_ft_transformer(
        model, X_tr_num, X_tr_cat, y_train_ps,
        X_v_num, X_v_cat, y_val_ps,
        lr=lr, n_epochs=50, batch_size=256,
        task='classification', device_=str(device))
    raw  = predict_ft_transformer(model, X_v_num, X_v_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    return -roc_auc_score(y_val_ps, prob)

study_rn = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_rn.optimize(resnet_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_rn = study_rn.best_params
print(f"Best Tabular ResNet params: {best_rn}")


Early stopping occurred at epoch 96 with best_epoch = 81 and best_val_auc = 0.58239

Early stopping occurred at epoch 35 with best_epoch = 20 and best_val_auc = 0.5849

Early stopping occurred at epoch 73 with best_epoch = 58 and best_val_auc = 0.5773

Early stopping occurred at epoch 70 with best_epoch = 55 and best_val_auc = 0.58089

Early stopping occurred at epoch 51 with best_epoch = 36 and best_val_auc = 0.5967
Stop training because you reached max_epochs = 100 with best_epoch = 96 and best_val_auc = 0.5669

Early stopping occurred at epoch 40 with best_epoch = 25 and best_val_auc = 0.62027

Early stopping occurred at epoch 62 with best_epoch = 47 and best_val_auc = 0.59539

Early stopping occurred at epoch 98 with best_epoch = 83 and best_val_auc = 0.59084


[W 2026-04-09 02:12:33,393] Trial 9 failed with parameters: {'n_d': 45, 'n_a': 25, 'n_steps': 7, 'gamma': 1.5467102793432796, 'lr': 0.00023426581058204064} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\cs_no\Envs\DSS5104\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\cs_no\AppData\Local\Temp\ipykernel_22232\1812464455.py", line 13, in tabnet_porto_objective
    m.fit(X_tr_sc, y_train_ps.astype(int),
  File "c:\Users\cs_no\Envs\DSS5104\lib\site-packages\pytorch_tabnet\abstract_model.py", line 258, in fit
    self._train_epoch(train_dataloader)
  File "c:\Users\cs_no\Envs\DSS5104\lib\site-packages\pytorch_tabnet\abstract_model.py", line 489, in _train_epoch
    batch_logs = self._train_batch(X, y)
  File "c:\Users\cs_no\Envs\DSS5104\lib\site-packages\pytorch_tabnet\abstract_model.py", line 527, in _train_batch
    output, M_loss = self.network(X)
  File "c:

KeyboardInterrupt: 

In [ ]:
print("Trials done:", len(study_rn.trials))
print("Best AUC:", study_rn.best_value)
print("Best params:", study_rn.best_params)

In [37]:
print("Training Tabular ResNet across seeds...")
rn_train_curves = {}
for seed in SEEDS:
    set_seed(seed)
    model = ResNetWithCatEmbeddings(
        n_num=n_num_ps, cat_cardinalities=cat_cards_ps,
        emb_dim=best_rn['emb_dim'],
        n_blocks=best_rn['n_blocks'], d_main=best_rn['d_main'],
        d_hidden=best_rn['d_hidden'],
        dropout_first=best_rn['dropout'], dropout_second=best_rn['dropout'],
    )
    model, tr_l, va_l = train_ft_transformer(
        model, X_tr_num, X_tr_cat, y_train_ps,
        X_v_num, X_v_cat, y_val_ps,
        lr=best_rn['lr'], n_epochs=100, batch_size=256,
        task='classification', device_=str(device))
    rn_train_curves[seed] = (tr_l, va_l)
    raw  = predict_ft_transformer(model, X_te_num, X_te_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    pred = (prob >= 0.5).astype(int)
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, pred, prob)
    all_results.append({'method': 'TabularResNet', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")

Training TabNet across seeds...


NameError: name 'best_tn' is not defined

## Model 2: FT-Transformer

In [ ]:
def ft_porto_objective(trial):
    d_token   = trial.suggest_categorical('d_token', [64, 128, 192])
    n_blocks  = trial.suggest_int('n_blocks', 1, 3)
    attn_drop = trial.suggest_float('attention_dropout', 0.0, 0.3)
    ffn_drop  = trial.suggest_float('ffn_dropout', 0.0, 0.3)
    lr        = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    set_seed(42)
    model = rtdl_revisiting_models.FTTransformer.make_baseline(
        n_num_features=n_num_ps,
        cat_cardinalities=cat_cards_ps,
        d_token=d_token,
        n_blocks=n_blocks,
        attention_dropout=attn_drop,
        ffn_d_hidden=int(d_token * 4 / 3),
        ffn_dropout=ffn_drop,
        residual_dropout=0.0,
        last_layer_query_idx=[-1],
        d_out=1,
    )
    model, _, _ = train_ft_transformer(
        model, X_tr_num, X_tr_cat, y_train_ps,
        X_v_num, X_v_cat, y_val_ps,
        lr=lr, n_epochs=50, batch_size=256,
        task='classification', device_=str(device))
    raw  = predict_ft_transformer(model, X_v_num, X_v_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    return -roc_auc_score(y_val_ps, prob)

study_ft = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_ft.optimize(ft_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_ft = study_ft.best_params
print(f"Best FT-Transformer params: {best_ft}")


In [ ]:
print("Training FT-Transformer across seeds...")
ft_train_curves = {}
for seed in SEEDS:
    set_seed(seed)
    model = rtdl_revisiting_models.FTTransformer.make_baseline(
        n_num_features=n_num_ps,
        cat_cardinalities=cat_cards_ps,
        d_token=best_ft['d_token'],
        n_blocks=best_ft['n_blocks'],
        attention_dropout=best_ft['attention_dropout'],
        ffn_d_hidden=int(best_ft['d_token'] * 4 / 3),
        ffn_dropout=best_ft['ffn_dropout'],
        residual_dropout=0.0,
        last_layer_query_idx=[-1],
        d_out=1,
    )
    model, tr_l, va_l = train_ft_transformer(
        model, X_tr_num, X_tr_cat, y_train_ps,
        X_v_num, X_v_cat, y_val_ps,
        lr=best_ft['lr'], n_epochs=100, batch_size=256,
        task='classification', device_=str(device))
    ft_train_curves[seed] = (tr_l, va_l)
    raw  = predict_ft_transformer(model, X_te_num, X_te_cat, str(device))
    prob = torch.sigmoid(torch.tensor(raw)).numpy()
    pred = (prob >= 0.5).astype(int)
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, pred, prob)
    all_results.append({'method': 'FT-Transformer', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Model 3: XGBoost

In [ ]:
def xgb_porto_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 8),
        'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':       trial.suggest_float('subsample', 0.6, 1.0),
        'scale_pos_weight': spw,
        'random_state':    42
    }
    set_seed(42)
    m = xgb.XGBClassifier(**params, verbosity=0, use_label_encoder=False,
                           eval_metric='auc')
    m.fit(X_tr_all, y_train_ps)
    prob = m.predict_proba(X_v_all)[:, 1]
    return -roc_auc_score(y_val_ps, prob)

study_xgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_xgb = study_xgb.best_params
print(f"Best XGBoost params: {best_xgb}")


In [ ]:
print("Training XGBoost across seeds...")
xgb_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = xgb.XGBClassifier(**best_xgb, scale_pos_weight=spw,
                           random_state=seed, verbosity=0,
                           use_label_encoder=False, eval_metric='auc')
    m.fit(X_tr_all, y_train_ps)
    preds = m.predict(X_te_all)
    probs = m.predict_proba(X_te_all)[:, 1]
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, preds, probs)
    all_results.append({'method': 'XGBoost', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    xgb_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Model 4: LightGBM

In [ ]:
def lgb_porto_objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 500),
        'num_leaves':    trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'random_state':  42, 'verbose': -1, 'is_unbalance': True
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr_all, y_train_ps)
    prob = m.predict_proba(X_v_all)[:, 1]
    return -roc_auc_score(y_val_ps, prob)

study_lgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(lgb_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_lgb = study_lgb.best_params
print(f"Best LightGBM params: {best_lgb}")


In [ ]:
print("Training LightGBM across seeds...")
lgb_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = lgb.LGBMClassifier(**best_lgb, random_state=seed, verbose=-1,
                            is_unbalance=True)
    m.fit(X_tr_all, y_train_ps)
    preds = m.predict(X_te_all)
    probs = m.predict_proba(X_te_all)[:, 1]
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, preds, probs)
    all_results.append({'method': 'LightGBM', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    lgb_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Model 5: Random Forest

In [ ]:
def rf_porto_objective(trial):
    params = {
        'n_estimators':    trial.suggest_int('n_estimators', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state':    42
    }
    set_seed(42)
    m = RandomForestClassifier(**params, class_weight='balanced', n_jobs=-1)
    m.fit(X_tr_all, y_train_ps)
    prob = m.predict_proba(X_v_all)[:, 1]
    return -roc_auc_score(y_val_ps, prob)

study_rf = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(rf_porto_objective, n_trials=N_OPTUNA_TRIALS)
best_rf = study_rf.best_params
print(f"Best RF params: {best_rf}")


In [ ]:
print("Training Random Forest across seeds...")
rf_model_last = None
for seed in SEEDS:
    set_seed(seed)
    m = RandomForestClassifier(**best_rf, random_state=seed,
                                class_weight='balanced', n_jobs=-1)
    m.fit(X_tr_all, y_train_ps)
    preds = m.predict(X_te_all)
    probs = m.predict_proba(X_te_all)[:, 1]
    acc, auc, gini, f1 = compute_porto_metrics(y_test_ps, preds, probs)
    all_results.append({'method': 'RandomForest', 'seed': seed,
                        'accuracy': acc, 'auc': auc, 'gini': gini, 'f1': f1})
    rf_model_last = m
    print(f"  Seed {seed}: Acc={acc:.4f}, AUC={auc:.4f}, Gini={gini:.4f}, F1={f1:.4f}")


## Results

In [ ]:
df_res = pd.DataFrame(all_results)
summary = df_res.groupby('method').agg(
    acc_mean=('accuracy', 'mean'),  acc_std=('accuracy', 'std'),
    auc_mean=('auc', 'mean'),       auc_std=('auc', 'std'),
    gini_mean=('gini', 'mean'),     gini_std=('gini', 'std'),
    f1_mean=('f1', 'mean'),         f1_std=('f1', 'std')
).round(4)

summary['Accuracy']      = summary['acc_mean'].astype(str)  + ' +/- ' + summary['acc_std'].astype(str)
summary['AUC-ROC']       = summary['auc_mean'].astype(str)  + ' +/- ' + summary['auc_std'].astype(str)
summary['Norm. Gini']    = summary['gini_mean'].astype(str) + ' +/- ' + summary['gini_std'].astype(str)
summary['F1']            = summary['f1_mean'].astype(str)   + ' +/- ' + summary['f1_std'].astype(str)
print(summary[['Accuracy', 'AUC-ROC', 'Norm. Gini', 'F1']])


## Visualizations

In [ ]:
methods = summary.index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(methods, summary['acc_mean'].values, yerr=summary['acc_std'].values,
               capsize=5, color='steelblue', alpha=0.8)
axes[0, 0].set_title('Accuracy')
axes[0, 0].tick_params(axis='x', rotation=30)

axes[0, 1].bar(methods, summary['auc_mean'].values, yerr=summary['auc_std'].values,
               capsize=5, color='darkorange', alpha=0.8)
axes[0, 1].set_title('AUC-ROC')
axes[0, 1].tick_params(axis='x', rotation=30)

axes[1, 0].bar(methods, summary['gini_mean'].values, yerr=summary['gini_std'].values,
               capsize=5, color='forestgreen', alpha=0.8)
axes[1, 0].set_title('Normalized Gini')
axes[1, 0].tick_params(axis='x', rotation=30)

axes[1, 1].bar(methods, summary['f1_mean'].values, yerr=summary['f1_std'].values,
               capsize=5, color='firebrick', alpha=0.8)
axes[1, 1].set_title('F1 Score')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.suptitle('Porto Seguro – Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Tabular ResNet training curves
fig, axes = plt.subplots(1, len(SEEDS), figsize=(15, 4))
for ax, seed in zip(axes, SEEDS):
    tr_l, va_l = rn_train_curves[seed]
    ax.plot(tr_l, label='Train')
    ax.plot(va_l, label='Val')
    ax.set_title(f'Seed {seed}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend()
plt.suptitle('Tabular ResNet Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# FT-Transformer training curves
fig, axes = plt.subplots(1, len(SEEDS), figsize=(15, 4))
for ax, seed in zip(axes, SEEDS):
    tr_l, va_l = ft_train_curves[seed]
    ax.plot(tr_l, label='Train')
    ax.plot(va_l, label='Val')
    ax.set_title(f'Seed {seed}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend()
plt.suptitle('FT-Transformer Training Curves', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# XGBoost feature importance (top 20)
if xgb_model_last is not None:
    fi = xgb_model_last.feature_importances_
    all_feat_names = num_cols_ps + cat_cols_filtered
    fi_df = pd.DataFrame({'feature': all_feat_names[:len(fi)], 'importance': fi})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(20)
    fi_df.plot.barh(x='feature', y='importance', figsize=(9, 7),
                    legend=False, color='teal')
    plt.title('XGBoost Top-20 Feature Importance')
    plt.tight_layout()
    plt.show()


In [ ]:
# LightGBM feature importance (top 20)
if lgb_model_last is not None:
    fi = lgb_model_last.feature_importances_
    all_feat_names = num_cols_ps + cat_cols_filtered
    fi_df = pd.DataFrame({'feature': all_feat_names[:len(fi)], 'importance': fi})
    fi_df = fi_df.sort_values('importance', ascending=True).tail(20)
    fi_df.plot.barh(x='feature', y='importance', figsize=(9, 7),
                    legend=False, color='mediumpurple')
    plt.title('LightGBM Top-20 Feature Importance')
    plt.tight_layout()
    plt.show()


## Analysis & Conclusions

### Summary
We benchmarked five models on the Porto Seguro safe-driver prediction task.

- The dataset is **severely imbalanced** (~3.6 % positive), so **Normalized Gini**
  (= 2·AUC − 1) is the key competition metric.
- **LightGBM** with `is_unbalance=True` and **XGBoost** with `scale_pos_weight`
  are well-suited for this imbalanced setting.
- **FT-Transformer** uses learned categorical embeddings, which can capture
  non-linear interactions between the many categorical features.
- **Tabular ResNet** uses skip connections to learn deep tabular representations.
- **Random Forest** with `class_weight='balanced'` offers a strong tree baseline.

### Observations
- Gini scores on **synthetic** data are near 0 because the synthetic labels are
  random (independent of features). Real data results will differ substantially.
- The preprocessing pipeline — replacing -1 sentinels, mode/median imputation,
  OrdinalEncoding — mirrors common Kaggle solutions for this competition.
- 3-seed evaluation reduces variance in reported Gini.

### Next Steps
- Feature engineering: polynomial interactions among `ps_reg` features.
- Calibrated probability outputs for better threshold tuning.
- Stacking / blending of LightGBM + XGBoost predictions.
- SHAP analysis to understand which feature groups drive predictions.
